# GeoAI Change Detection Notebook

**Demonstrates STAC querying + lazy COG loading + simple NDVI-based land cover change detection**

This notebook is designed to run in **Google Colab**.

It uses open Sentinel-2 L2A data via public STAC catalogs and shows how to work with Cloud Optimized GeoTIFFs (COGs) efficiently.

## 1. Install Dependencies

In [ ]:
!pip install -q pystac-client planetary-computer rasterio rioxarray geopandas matplotlib leafmap

## 2. Imports and Setup

In [ ]:
import pystac_client
import planetary_computer
import rasterio
from rasterio.plot import show
import rioxarray as rxr
import geopandas as gpd
import matplotlib.pyplot as plt
from datetime import datetime

print("✅ Libraries imported successfully!")

## 3. Define Area of Interest (AOI) and Time Range

Change the `bbox` and dates as needed. The example below uses a small area in the central US for fast testing.

In [ ]:
# Example bounding box: [min_lon, min_lat, max_lon, max_lat]
# You can get bbox from geojson.io or Leaflet
bbox = [-100.0, 35.0, -99.5, 35.5]   # Small test area

# Time ranges (before and after period)
time_before = "2023-06-01/2023-06-30"
time_after  = "2023-09-01/2023-09-30"

print("AOI bounding box:", bbox)

## 4. Query STAC Catalog

In [ ]:
# Open the Planetary Computer STAC catalog (recommended for Sentinel-2 L2A)
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace   # Required for signed COG URLs
)

# Search for scenes before the event
search_before = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=time_before,
    query={"eo:cloud_cover": {"lt": 30}},
    limit=5
)

items_before = list(search_before.items())
print(f"Found {len(items_before)} scenes before")

# Search for scenes after the event
search_after = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=time_after,
    query={"eo:cloud_cover": {"lt": 30}},
    limit=5
)

items_after = list(search_after.items())
print(f"Found {len(items_after)} scenes after")

## 5. Load COGs Lazily and Compute NDVI

We load only the needed bands (B04 = Red, B08 = NIR) using `rioxarray`.

In [ ]:
def load_band(item, band_name):
    """Load a single band COG lazily"""
    href = item.assets[band_name].href
    return rxr.open_rasterio(href, masked=True).squeeze()

# Load Red and NIR for before and after (using first item for simplicity)
if items_before and items_after:
    red_before = load_band(items_before[0], "B04")
    nir_before = load_band(items_before[0], "B08")
    
    red_after = load_band(items_after[0], "B04")
    nir_after = load_band(items_after[0], "B08")
    
    # Compute NDVI
    ndvi_before = (nir_before - red_before) / (nir_before + red_before)
    ndvi_after  = (nir_after - red_after) / (nir_after + red_after)
    
    print("✅ NDVI calculated for before and after periods")
else:
    print("No items found. Try adjusting bbox or dates.")

## 6. Visualize NDVI

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

ndvi_before.plot(ax=axs[0], cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axs[0].set_title("NDVI Before")

ndvi_after.plot(ax=axs[1], cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axs[1].set_title("NDVI After")

plt.tight_layout()
plt.show()

## 7. Simple Change Detection (NDVI Difference)

Positive values = increase in vegetation, negative = decrease

In [ ]:
ndvi_diff = ndvi_after - ndvi_before

plt.figure(figsize=(8, 6))
ndvi_diff.plot(cmap="BrBG", vmin=-0.4, vmax=0.4)
plt.title("NDVI Change (After - Before)")
plt.show()

# Optional: Save difference as GeoTIFF (small file)
ndvi_diff.rio.to_raster("ndvi_change.tif")
print("NDVI difference saved as ndvi_change.tif")

## ✅ Key Achievements Demonstrated

- Efficient querying of Sentinel-2 data using **public STAC catalogs**
- Lazy loading of **Cloud Optimized GeoTIFFs (COGs)** — only the needed data is accessed
- Automated feature extraction via **NDVI differencing** for land cover change detection
- Reproducible workflow that can scale to larger areas or more advanced models

This notebook serves as the analytical backend for the interactive Leaflet web map in the `/webmap` folder.